# CPU FP32 / CPU INT8 / FPGA-3 / FPGA-4 整网对照

在电脑解压，把整个 `benchmark_compare` 文件夹放入现有 `full_model_test/evaluation/`。由你手动运行，不安装依赖、不改原模型或旧结果。
计算范围：同一板卡、1线程、同一完整原始Fbank→完整CAM++→L2→已注册余弦评分。包含CPU剩余运算、参数地址切换、DDR复制/缓存同步，排除采集/读音频/Fbank/逐层参考/首次加载。初始化单独记录；运行库已预先载入。
FP32及INT8完整CPU基线都开启ORT图优化；第三/四版沿用已验证的CPU分段图及原优化策略。四版都使用相同零显式padding AveragePool兼容语义。

先关闭其他正在使用FPGA的Notebook。本Notebook直接在持久kernel运行，禁止另开并行推理。若IP超时/运行中中断，保持kernel直到板卡断电重启，之后再重新开始。


In [ ]:
from pathlib import Path
import sys, uuid
sys.dont_write_bytecode=True
WORK=Path.cwd().resolve()
assert WORK.name=='benchmark_compare' and (WORK/'compare_benchmark.py').is_file(), '请从full_model_test/evaluation/benchmark_compare打开'
sys.path.insert(0,str(WORK))
from compare_benchmark import setup, hardware_check, integration_check, benchmark
RUN=WORK/'results'/('run_'+uuid.uuid4().hex)
INCLUDE_FP32=True
REPEATS=20
WARMUP=3
setup(RUN)
print('结果目录：',RUN)


## 1. 第四版独立硬件门控

会加载新双IP Overlay，运行冒烟和完整FCM；已有本包同指纹PASS记录时可以复用。必须出现V4_HARDWARE_READY。


In [ ]:
hardware_report=hardware_check(RUN)


## 2. 整网接入验证（不计时）

检查CPU分图重建、FPGA逐层INT8/INT32，以及三段完整音频的CPU/V3/V4向量和分数。原QDQ漂移不自动改判通过；只有硬件逐点一致及分图重建通过才允许工程时延测试。出现INTEGRATION_READY后继续。


In [ ]:
integration_report=integration_check(RUN, include_fp32=INCLUDE_FP32)


## 3. 正式模型测速

每种模式、每条查询预热后测20次，分两轮反序执行；一次只保留一个运行时。逐层整数参考、调试输出和审计均关闭。全部模型推理及主机调用开销仍在计时内。


In [ ]:
performance_report=benchmark(RUN, repeats=REPEATS, warmup=WARMUP, include_fp32=INCLUDE_FP32)


返回最后的MODEL_LATENCY、SPEEDUP_COMPARE、COMPARE_BENCHMARK_COMPLETED，以及本次results/run编号。summary.csv/summary.md可直接看中位数、P95和倍率；raw_samples.json保留全部样本。倍率小于1即更慢，按实测报告。精度/评分漂移单独报告，不输出未经校准的身份通过或拒绝，也不将模型时延写成音频端到端时延。
